In [ ]:
from xmlrpc.server import SimpleXMLRPCServer, SimpleXMLRPCRequestHandler
import xmlrpc.client as xmlrpclib
import datetime
import pandas as pd
import numpy as np
import pickle


class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("127.0.0.1", 8008),
                            requestHandler=RequestHandler,
                            allow_none=True)

server.register_introspection_functions()

stats_server = xmlrpclib.ServerProxy("http://localhost:8018", allow_none=True)

# Добавление в лог через сервер
def add_log(log_line):
  try:
    stats_server.add_log(log_line)
  except Exception:
       # Стат-сервер недоступен — не мешаем основной работе
       pass
  return True

# Тест
def ping():
    add_log('ping')
    return True
server.register_function(ping, 'ping')

# Время сервера
def now():
    add_log('now')
    return datetime.datetime.now()
server.register_function(now, 'now')

# Отображение строкового вида, типа и значений
def show_type(arg):
    add_log('type')
    return (str(arg), str(type(arg)), arg)
server.register_function(show_type, 'type')

# Сумма
def test_sum(a, b):
    add_log('sum')
    return a + b
server.register_function(test_sum, 'sum')

# Степень
def test_pow(a, b):
    add_log('pow')
    return a**b
server.register_function(test_pow, 'pow')

# Проверка нахождения клиента в черном списке c использованием Pandas Data Frame
#def black_list_check(sname):
#    frame = pd.read_csv('bad_boys2.csv', header=0, sep=',', encoding='utf8')
#    exist = any(frame['Surname'] == sname)
#    if (exist == True):
#        return sname + ": "+ "bad_boy"
#    else:
#        return sname + ": "+ "good_boy"

def black_list_check(surname, name, patronym, birth):
    add_log('black') # добавляем лог
    # Приводим строки к нижнему регистру
    input_data = f"{surname} {name} {patronym} ({birth})"
    surname = surname.strip().lower()
    name = name.strip().lower()
    patronym = patronym.strip().lower()
    birth = birth.strip()

    # Чтение CSV
    frame = pd.read_csv('bad_boys2.csv', header=0, sep=',', encoding='utf8')

    # Приводим строки БД к нижнему регистру
    frame['Surname_norm']  = frame['Surname'].astype(str).str.strip().str.lower()
    frame['Name_norm']     = frame['Name'].astype(str).str.strip().str.lower()
    frame['Patronym_norm'] = frame['Patronym'].astype(str).str.strip().str.lower()
    frame['Birth_norm']    = frame['Birth'].astype(str).str.strip()

    # Оотбираем записи с совпадающей датой рождения
    candidates = frame[frame['Birth_norm'] == birth]

    exact_matches = [] #переменная для сохранения точного совпадающего поля
    similar_matches = [] #переменная для сохранения не точно совпадающих полей

    for _, row in candidates.iterrows():
        ref_surname  = row['Surname_norm']  #разделяем массив на части ФИО
        ref_name     = row['Name_norm']
        ref_patronym = row['Patronym_norm']

        # 1- Суммарная разница длин по ФИО
        diff = (abs(len(surname)  - len(ref_surname)) +
                abs(len(name)     - len(ref_name)) +
                abs(len(patronym) - len(ref_patronym)))

        # Проверяем разницу после длинн
        if diff > 1:
            continue

        # 2 - Посимвольное сравнение по общим позициям
        pairs = [
            (surname,  ref_surname),
            (name,     ref_name),
            (patronym, ref_patronym),
        ]
        for inp, ref in pairs:
            for i in range(min(len(inp), len(ref))):
                if inp[i] != ref[i]:
                    diff += 1
                    if diff > 1:
                        break
            if diff > 1:
                break

        # 3 - классификация
        if diff == 0:
            exact_matches.append(row)
        elif diff == 1:
            similar_matches.append(row)



    if exact_matches:
      r = exact_matches[0]
      return f"{r['Surname']} {r['Name']} {r['Patronym']} ({r['Birth']}): yes"
    elif similar_matches:
      similar_list = [
        f"{row['Surname']} {row['Name']} {row['Patronym']} ({row['Birth']})"
        for row in similar_matches
        ]
      return f"{similar_list}: similar"
    else:
       return f"{input_data}: no"


server.register_function(black_list_check, 'black_list_check')



# Бинарная передача данных
def send_back_binary(bin_data):
    add_log('binar')
    data = bin_data.data
    return xmlrpclib.Binary(data)
server.register_function(send_back_binary, 'send_back_binary')

# Инверсия цвета
# На вход изображение RGB размерности (M, N, 3) со значениями 0-255
#def send_back_inversion(bin_data):
#     img_arr = pickle.loads(bin_data.data)
#
#     height = img_arr.shape[0]
#     width = img_arr.shape[1]
#
#     for i in range(height):
#          for j in range(width):
#              img_arr[i][j][0] = 255 - img_arr[i][j][0]
#              img_arr[i][j][1] = 255 - img_arr[i][j][1]
#              img_arr[i][j][2] = 255 - img_arr[i][j][2]
#
#     pimg = pickle.dumps(img_arr)
#     return xmlrpclib.Binary(pimg)
#server.register_function(send_back_inversion, 'send_back_inversion')

def send_back_inversion(bin_data):
    add_log('invers')
    img = pickle.loads(bin_data.data) # Распаковка в numpy array

    inverted_img = 255 - img

    return xmlrpclib.Binary(pickle.dumps(inverted_img))

server.register_function(send_back_inversion, 'send_back_inversion')


def binarize(bin_data, threshold):
    add_log('binariz')
    if not (1 <= threshold <= 255):
        raise ValueError("Порог должен быть в диапазоне от 1 до 255")

    img = pickle.loads(bin_data.data)

    # Бинаризация
    # Создаём пустой массив
    #НАДО ИСПРАВИТЬ - БИНОРИЗАЦИЯ НЕ ПО ВСЕМ КАНАЛАМ СИНХРОНИЗИРОВАННА?
    if img.dtype != np.uint8:
      img = np.clip(img, 0 ,255).astype(np.uint8)
    if img.ndim == 3:  #если изображение цветное, вывод будет вывод инф-ции по 3-м каналам
        gray = img.astype(np.float32).mean(axis=2)
        mask = gray > threshold
        bin_img = np.zeros_like(img, dtype = np.uint8)
        bin_img[mask] = 255
    else:             #если изображение ЧБ вывод по одному каналу
        gray = img.astype(np.float32)
        mask = gray > threshold
        bin_img = np.zeros_like(img, dtype = np.uint8)
        bin_img[mask] = 255

    return xmlrpclib.Binary(pickle.dumps(bin_img))

server.register_function(binarize, 'binarize')

def binarize_percent(bin_data, threshold):
    add_log('binariz_p') #добавляем лог
    if not (1 <= threshold <= 255):
        raise ValueError("Порог должен быть в диапазоне от 1 до 255")

    img = pickle.loads(bin_data.data)



    if img.dtype != np.uint8:
      img = np.clip(img, 0 ,255).astype(np.uint8)
    if img.ndim == 3:  #если изображение цветное, вывод будет вывод инф-ции по 3-м каналам
        gray = img.astype(np.float32).mean(axis=2)
        mask = gray > threshold
        bin_img = np.zeros_like(img, dtype = np.uint8)
        bin_img[mask] = 255

        total = img.size
        percent = (img > threshold).sum() / total * 100.0
        return f"Gray Chenal - {percent:.2f}%"
    else:             #если изображение ЧБ вывод по одному каналу
        gray = img.astype(np.float32)
        mask = gray > threshold
        bin_img = np.zeros_like(img, dtype = np.uint8)
        bin_img[mask] = 255

        total = img.size
        percent = (img > threshold).sum() / total * 100.0
        return f"Gray Chenal - {percent:.2f}%"

server.register_function(binarize_percent, 'binarize_percent')

# --- Пункт 3: Разворот изображения относительно вертикали ---
def flip_vertical(bin_data):
    add_log('flip')
    img = pickle.loads(bin_data.data)


    # Делаем срез; переворачиваем столбцы слева направо
    if len(img.shape) == 3: # (M, N, 3) или (M, N, 1)
        flipped_img = img[:, ::-1, :]
    else: # (M, N)
        flipped_img = img[:, ::-1]

    return xmlrpclib.Binary(pickle.dumps(flipped_img))

server.register_function(flip_vertical, 'flip_vertical')

def get_log(event_type=None, time_from=None, time_to=None):
    try:
        return stats_server.get_log(event_type, time_from, time_to)
    except Exception as e:
        return f"Stats server unavailable: {e}"

server.register_function(get_log, 'get_log')






print("Listening on port 8008...")
server.serve_forever()


Listening on port 8008...


127.0.0.1 - - [19/Oct/2026 16:17:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:39] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:40] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:44] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:45] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:17:49] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:20:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:20:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:20:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/2026 16:20:33] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [19/Oct/202